<a href="https://colab.research.google.com/github/marcio-antonio/ROSS_testing_marcio/blob/main/ROSS_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%capture
# 1. Run this cell first to install ROSS in Colab
!pip install ross-rotordynamics

In [3]:
%%capture
import ross as rs
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

print("ROSS version:", rs.__version__)

In [4]:
steel = rs.Material(name="Steel", rho=7810, E=211e9, G_s=81.2e9)
# Creating a list of shaft elements
L = 0.25
i_d = 0
o_d = 0.05
N_elem = 6

shaft_elements = [
    rs.ShaftElement(
        L=L,
        idl=i_d,
        odl=o_d,
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for _ in range(N_elem)
]

In [18]:
disk = rs.DiskElement(
    n=2,
    m=32.58,
    Ip=0.178,
    Id=0.329,
    tag="Disk"
)
disk2 = rs.DiskElement(
    n=3,
    m=32.58,
    Ip=0.178,
    Id=0.329,
    tag="Disk2"
)

In [6]:
bearing0 = rs.BearingElement(
    n=0,
    kxx=[0.5e6, 1.0e6, 2.5e6],
    kyy=[1.5e6, 2.0e6, 3.5e6],
    cxx=[0.5e3, 1.0e3, 1.5e3],
    frequency=[0, 1000, 2000],
)
bearing1 = rs.BearingElement(
    n=5,
    kxx=[0.5e6, 1.0e6, 2.5e6],
    kyy=[1.5e6, 2.0e6, 3.5e6],
    cxx=[0.5e3, 1.0e3, 1.5e3],
    frequency=[0, 1000, 2000],
)
bearings = [bearing0, bearing1]

In [19]:
# Force Plotly to target Google Colab's renderer
pio.renderers.default = 'colab'

# Re-assemble your rotor
rotor = rs.Rotor(shaft_elements=shaft_elements, disk_elements=[disk, disk2], bearing_elements=bearings)

# Generate and force display
fig = rotor.plot_rotor()
fig.show()

**Station = node**\
When you define elements in your code (like a shaft element), you specify its length. ROSS takes your entire physical geometry and cuts it into segments. The boundaries between these segments are numbered points **(0, 1, 2, 3...)**.\
In general Finite Element Analysis (FEA), these are called nodes.\
In rotordynamics and ROSS specifically, they are called stations. \When you attach a disc or a bearing in your code, you must specify the exact n (station number) where it sits.

**Physical Deformation Coordinates**\
Each station in ROSS tracks 4 distinct physical movements (Degrees of Freedom) to capture 3D bending:

1.   Displacement along the X-axis (horizontal bending)
2.   Displacement along the Y-axis (vertical bending)
3.   Rotation around the X-axis (tilt)
4.   Rotation around the Y-axis (tilt)

**The Matrix setup**\
When you build your rotor in ROSS, the code generates massive matrices representing the total Mass **M**, Stiffness **K**, and Gyroscopic effects **G** of your entire system. If you have 10 stations, and each station has 4 Degrees of Freedom (DOFs), your system matrix has a total of 40 dimensions.\
**Eigenvalues (The Frequencies)** \
When you run .run_modal(), the solver cracks open that matrix. It finds a set of special numbers called eigenvalues. Each individual eigenvalue represents one specific natural frequency (and its damping/Log Dec ratio). If your system has 40 DOFs, the math will yield up to 40 individual eigenvalues.\
**Eigenvectors (The Shapes)**  
Complete Deformed Shape. One shape for each dimension (40) : Those 40 numbers map out the entire physical geometry at once.\  
The first 4 numbers tell ROSS how Station 0 moves. The next 4 numbers tell ROSS how Station 1 moves....and so on, all the way to Station 9.





1.   The Eigenvalue belongs to the whole machine (the rope spinning at 2500 Hz).
2.   The Eigenvector belongs to the whole machine (the full wave shape of the rope).
3.   The Stations just own individual rows of data inside that single master eigenvector.



In [20]:
campbell = rotor.run_campbell(speed_range=np.linspace(0, 1000))
campbell.plot()

*   titles (Forward, Backward, Mixed, Axial, and Torsional) do not describe external constraints or wave propagation velocities. Instead, they describe how the shaft physically bends and whirls in space at that specific natural frequency.
*    Log Dec δ represents how fast a free vibration dies out over time. Low Log Dec (Red = 0.0): This indicates very little damping. A value here means that if an excitation hits this frequency, the vibrations will ring out for a long time. If it drops below **0**, the system becomes unstable and can tear itself apart.

*   The lines at the bottom (around 1,500 RPM) are colored dark red/brown. This indicates that these specific lower natural frequencies have low damping parameters.The line tracking around 2,500 RPM shifts into light blue/blue. This indicates a highly stable, well-damped mode shape.Because the bearings supply different levels of oil-film or mechanical damping to different bending directions, every distinct natural frequency will naturally exhibit a completely different Log Dec value.






*   High Log Dec is Great: A Log Dec near 1.0 means that if the rotor passes through or experiences vibration at that specific frequency, the system is excellent at absorbing and dissipating that energy. The vibration amplitude will remain small and safe.
*   The "Operating Speed" Target: While a high Log Dec mode is fantastic, engineers do not usually design the regular operating speed (the steady RPM of the motor) to sit right on top of a natural frequency line—even a well-damped one.

*   The Goal: The ideal design goal is to place your steady-state operating speed in a wide, clear "valley" between the critical speeds (the "X" intersections).



In [ ]:
# 1. Run the modal analysis (assuming your 'rotor' object is already defined)
# This calculates natural frequencies and mode shapes across your speed range
modal = rotor.run_modal(speed=2500)  # Input your desired operating speed in RPM

# 2. Plot the specific mode shape you want to see
# mode=0 is the 1st natural frequency, mode=1 is the 2nd, etc.
fig = modal.plot_mode_2d(mode=1)

# 3. Force display of the interactive 3D canvas
fig.show()